## Access workspace bucket data with temporary AWS credentials

When logged into the ADE, temporary s3 credentials can be issued using the maap-py function `maap.aws.workspace_bucket_credentials()`

This command issues a set of AWS credentials that grant full read/write access to your own user folder within the workspace bucket, as well as any additional S3 buckets your organization has been granted access to.

The response contains:
- `credentials` — temporary AWS credentials (`aws_access_key_id`, `aws_secret_access_key`, `aws_session_token`, `expires_at`)
- `authorized_s3_paths` — an array of accessible paths, each with `bucket`, `prefix`, `uri`, `type` (`workspace` or `org`), and `access` (`read_write` or `read_only`)

### 1. Retrieve temporary credentials

```python
import json
from maap.maap import MAAP
maap = MAAP()

resp = maap.aws.workspace_bucket_credentials()
print(json.dumps(resp, indent=2))
>>> {
  "credentials": {
    "aws_access_key_id": "...",
    "aws_secret_access_key": "...",
    "aws_session_token": "...",
    "expires_at": "2025-03-03T18:00:00Z"
  },
  "authorized_s3_paths": [
    {
      "bucket": "maap-ops-workspace",
      "prefix": "maap_user",
      "uri": "s3://maap-ops-workspace/maap_user",
      "type": "workspace",
      "access": "read_write"
    },
    {
      "bucket": "shared-project-bucket",
      "prefix": "team-data",
      "uri": "s3://shared-project-bucket/team-data",
      "type": "org",
      "access": "read_write"
    },
    {
      "bucket": "public-reference-data",
      "prefix": "smap/v9",
      "uri": "s3://public-reference-data/smap/v9",
      "type": "org",
      "access": "read_only"
    }
  ]
}
```

### 2. Create a boto3 session from the credentials

```python
import boto3

creds = resp["credentials"]
session = boto3.Session(
    aws_access_key_id=creds["aws_access_key_id"],
    aws_secret_access_key=creds["aws_secret_access_key"],
    aws_session_token=creds["aws_session_token"],
)
s3 = session.client("s3")
```

### 3. Working with your workspace bucket

The workspace path is always the first entry in `authorized_s3_paths`. Use the `bucket` and `prefix` fields directly:

```python
workspace = resp["authorized_s3_paths"][0]
bucket = workspace["bucket"]
prefix = workspace["prefix"]

# List objects
response = s3.list_objects_v2(Bucket=bucket, Prefix=prefix + "/", MaxKeys=10)
for obj in response.get("Contents", []):
    print(obj["Key"])

# Download a file
s3.download_file(Bucket=bucket, Key=f"{prefix}/my_file.csv", Filename="my_file.csv")

# Upload a file
s3.upload_file(Filename="local_results.csv", Bucket=bucket, Key=f"{prefix}/local_results.csv")
```

### 4. Working with organization shared buckets

Additional org-granted buckets appear as extra entries. Each entry tells you whether it is `read_write` or `read_only`:

```python
for path in resp["authorized_s3_paths"]:
    print(f"{path['uri']}  ({path['access']})")

# Access a specific org bucket
shared = resp["authorized_s3_paths"][1]
shared_bucket = shared["bucket"]
shared_prefix = shared["prefix"]

# List files
response = s3.list_objects_v2(Bucket=shared_bucket, Prefix=shared_prefix + "/", MaxKeys=10)
for obj in response.get("Contents", []):
    print(obj["Key"])

# Download a file
s3.download_file(Bucket=shared_bucket, Key=f"{shared_prefix}/shared_dataset.tif", Filename="shared_dataset.tif")

# Upload a file (only works if access is "read_write")
if shared["access"] == "read_write":
    s3.upload_file(Filename="my_output.tif", Bucket=shared_bucket, Key=f"{shared_prefix}/my_output.tif")
```